In [14]:
import pandas as pd
import numpy as np
import re

In [15]:
df = pd.read_csv(r"C:\Users\Junayed\pandas_prac\Aug_15\messy_campaigns.csv")

In [16]:
df.shape

(48, 16)

In [17]:
df.dtypes

CampaignID          str
CampaignName        str
Channel             str
Country             str
StartDate           str
EndDate             str
Impressions       int64
Clicks            int64
CTR                 str
Spend               str
Revenue             str
ROI                 str
LandingPageURL      str
Device              str
Status              str
Notes               str
dtype: object

In [18]:
df.head(3)

,CampaignID,CampaignName,Channel,Country,StartDate,EndDate,Impressions,Clicks,CTR,Spend,Revenue,ROI,LandingPageURL,Device,Status,Notes
0,CMP3001,Fall Sale Blitz,email,France,2023-09-01,2023-09-06,11310,444,3.93%,1283.19,1904.77,0.48,https://example.com/fall-sale,mobile,PAUSED,NaN
1,CMP3002,Back to School,Social,Spain,2023-09-02T08:00:00+00:00,2023-09-07T08:00:00+00:00,12531,568,4.53%,746.37,381.49,-0.49,https://shop.example.com/promo,Desktop,Paused,NaN
2,CMP3003,Q3 Retargeting,Email,Spain,2023-09-03T08:00:00-08:00,2023-09-08T08:00:00-08:00,49738,1306,2.63%,736.45,477.35,-0.35,htp://example.com/typo,Tablet,active,NaN


## Acquisition Channel Casing & Whitespace Normalization

**Issue Identified:**  
The `Channel` column contains mixed casing variations (e.g., `"email"`, `"Email"`, `"paid search"`, `"Paid Search"`, `"PAID SEARCH"`, `"Display"`, `"social"`, `"Social"`), which causes identical marketing channels to be treated as distinct categories during aggregation and attribution analysis.

**Fix Applied:**  
1. **String Type Enforcement:** Converted entries to string data types using `.astype(str)`.
2. **Whitespace Trimming:** Stripped any accidental leading and trailing whitespace using `.str.strip()`.
3. **Title Casing Applied:** Applied `.str.title()` to standardize all channel names into uniform Title Case format (e.g., standardizing all variants to `"Paid Search"`, `"Email"`, `"Social"`, and `"Display"`).

In [19]:
df["Channel"] = df["Channel"].astype(str).str.strip().str.title()

In [20]:
df["Country"].isna().sum()

np.int64(1)

## Multi-Timezone & Mixed Format Parsing

**Issue Identified:**  
The `StartDate` and `EndDate` columns contain heterogeneous datetime strings—including date-only strings (`YYYY-MM-DD`), ISO 8601 strings, and varying UTC timezone offsets (`+00:00`, `-08:00`, `+01:00`, `+09:00`, `-05:00`). Unifying them without explicit handling can cause comparison errors and misaligned durations.

**Fix Applied:**  
1. **Mixed Format Parsing:** Applied `pd.to_datetime(..., format='mixed')` to reliably ingest varying string formats across rows.
2. **UTC Normalization:** Set `utc=True` to convert and align all timestamps to a shared Universal Time Coordinate (UTC) standard, enabling valid cross-row chronological comparisons.

In [21]:
df["StartDate"] = pd.to_datetime(df["StartDate"], format='mixed', utc=True)
df["EndDate"] = pd.to_datetime(df["EndDate"], format='mixed', utc=True)

df[["StartDate", "EndDate"]].head(5)

,StartDate,EndDate
0,2023-09-01 00:00:00+00:00,2023-09-06 00:00:00+00:00
1,2023-09-02 08:00:00+00:00,2023-09-07 08:00:00+00:00
2,2023-09-03 16:00:00+00:00,2023-09-08 16:00:00+00:00
3,2023-09-04 07:00:00+00:00,2023-09-09 07:00:00+00:00
4,2023-09-04 23:00:00+00:00,2023-09-09 23:00:00+00:00


## Identifying Inverted Date Ranges

**Issue Identified:**  
Some records contain inverted timelines where the campaign/stay conclusion timestamp (`EndDate`) occurs chronologically before its initiation timestamp (`StartDate`):
$$\text{EndDate} < \text{StartDate}$$

**Fix & Audit Applied:**  
1. **Anomaly Identification:** Created a boolean condition `invalid_stay` to detect records violating logical date ordering.
2. **Review Flagging:** Updated `NeedsReview` to flag anomalous records for manual audit and verification.

In [22]:
invalid_stay = df["EndDate"] < df["StartDate"]
df.loc[invalid_stay, ["CampaignID", "CampaignName", "Country", "StartDate", "EndDate"]]

df.loc[invalid_stay, "NeedsReview"] = invalid_stay

In [23]:
df.loc[invalid_stay, ["CampaignID", "CampaignName", "Country", "StartDate", "EndDate", "Notes"]]

,CampaignID,CampaignName,Country,StartDate,EndDate,Notes
24,CMP3025,Email Reactivation,France,2023-09-10 08:00:00+00:00,2023-09-01 08:00:00+00:00,End before start?


## Numerical Standardization for `Spend`

**Issue Identified:**  
The `Spend` column contains inconsistent numeric string formats, including:
* Standard decimal points (`1283.19`, `746.37`)
* European comma decimal separators (`601,38`, `775,01`)
* Comma thousands separators (`1,498`, `1,016`)

**Fix Applied:**  
1. **Separator Disambiguation:** Differentiated between European comma decimals (2 trailing digits) and thousands separators (3 trailing digits).
2. **String Sanitization:** Handled cases with mixed punctuation and stripped surrounding whitespace.
3. **Numeric Cast:** Converted cleaned string outputs into `float64` data types using `.apply()` with error-safe float coercion.

In [24]:
def clean_spend(val):
    if pd.isna(val):
        return np.nan

    val = str(val).strip()

    if "," in val and "." in val:
        if val.find(",") < val.find("."):
            val = val.replace(",", "")
        else:
            val = val.replace(".", "").replace(",", ".")
    elif "," in val:
        parts = val.split(",")
        if len(parts[-1]) == 2:
            val = val.replace(",", ".")
        elif len(parts[-1]) == 3:
            val = val.replace(",", "")
        else:
            val = val.replace(",", ".")
    try:
        return float(val)
    except ValueError:
        return np.nan

df["Spend"] = df["Spend"].apply(clean_spend)
df["Spend"]

0     1283.19
1      746.37
2      736.45
3     1643.44
4      418.39
5      601.38
6     2418.60
7     1194.63
8      987.96
9        0.00
10    1886.74
11     775.01
12    2043.91
13    1498.00
14    1252.02
15    1287.68
16     195.74
17    2650.29
18    1785.82
19    1801.38
20    2721.54
21    2369.13
22    1219.03
23    2245.39
24    1223.91
25     771.90
26     340.76
27    1601.07
28    2979.99
29    2055.69
30    2001.43
31    1016.00
32    2165.66
33     823.47
34    2892.93
35    2163.43
36    2239.95
37    1942.65
38    2543.85
39    1168.04
40    1194.63
41    2440.78
42    1224.55
43     961.44
44    2043.91
45     770.75
46    2704.05
47    1839.23
Name: Spend, dtype: float64

## Handling Spreadsheet Formula Errors in `Revenue`

**Issue Identified:**  
The `Revenue` column contains spreadsheet error codes (`"#REF!"`, `"#N/A"`, `"#VALUE!"`) alongside numeric values, preventing standard mathematical operations and statistical modeling.

**Fix Applied:**  
1. **Error Code Replacement:** Sanitized leading/trailing whitespace and explicitly replaced recognized spreadsheet calculation error artifacts (`["#REF!", "#N/A", "#VALUE!"]`) with `np.nan`.
2. **Numeric Type Coercion:** Applied `pd.to_numeric(..., errors='coerce')` to cast valid numerical string entries into `float64` format while safely coercing any remaining unparseable entries to `NaN`.

In [25]:
df["Revenue"] = df["Revenue"].astype(str).str.strip().replace(["#REF!", "#N/A", "#VALUE!"], np.nan)
df["Revenue"] = pd.to_numeric(df["Revenue"], errors='coerce')
df["Revenue"]

0      1904.77
1       381.49
2       477.35
3      4813.15
4      1568.04
5       378.86
6          NaN
7      2024.07
8      2169.15
9      1503.25
10     2136.49
11      845.09
12     7450.95
13     2718.41
14         NaN
15     1412.85
16      193.50
17     6903.33
18     1763.65
19     4960.11
20     2632.77
21     1780.67
22         NaN
23     7072.30
24     3040.56
25      807.88
26      887.38
27     3942.42
28     2612.32
29     6104.94
30     1295.21
31     1318.15
32     7832.55
33     1798.62
34    11462.46
35     1768.48
36     4744.71
37     5605.92
38     2637.78
39     4429.86
40     2024.07
41     9205.71
42     2503.38
43     2008.80
44     7450.95
45     1936.21
46     8421.08
47     4912.56
Name: Revenue, dtype: float64

## Handling `#DIV/0!` in `CTR` and `ROI`

**Issue Identified:**  
* The `CTR` (Click-Through Rate) column contains spreadsheet division-by-zero error strings (`"#DIV/0!"`) and percentage signs (`"%"`) stored as object/string data types.
* The `ROI` (Return on Investment) column contains division-by-zero errors (`"#DIV/0!"`), preventing direct numerical calculations and statistical modeling.

**Fix Applied:**  
1. **Division-by-Zero Handling:** Replaced `"#DIV/0!"` artifacts in both `CTR` and `ROI` with `np.nan` using `.replace()`.
2. **Percentage Sign Removal:** Stripped percentage signs (`"%"`) from string-formatted CTR records using `.str.replace("%", "", regex=False)`.
3. **Numeric Type Coercion:** Cast both columns to numeric (`float64`) using `pd.to_numeric(..., errors="coerce")` to facilitate performance analysis and visualization.

In [26]:
df["CTR"] = df["CTR"].replace("#DIV/0!", np.nan)
df["CTR"] = pd.to_numeric(df["CTR"].astype(str).str.replace("%", "", regex=False), errors="coerce")

df["ROI"] = df["ROI"].replace("#DIV/0!", np.nan)
df["ROI"] = pd.to_numeric(df["ROI"], errors="coerce")

df[["Impressions", "Clicks", "CTR", "Spend", "Revenue", "ROI"]].head(10)

,Impressions,Clicks,CTR,Spend,Revenue,ROI
0,11310,444,3.93,1283.19,1904.77,0.48
1,12531,568,4.53,746.37,381.49,-0.49
2,49738,1306,2.63,736.45,477.35,-0.35
3,0,0,NaN,1643.44,4813.15,1.93
4,49063,3298,6.72,418.39,1568.04,2.75
5,26802,1177,4.39,601.38,378.86,-0.37
6,14168,809,5.71,2418.60,NaN,1.80
7,31716,1558,4.91,1194.63,2024.07,0.69
8,30077,1356,4.51,987.96,2169.15,1.70
9,47560,2859,6.01,0.00,1503.25,NaN


## ROI Recalculation & Discrepancy Detection

**Objective:**  
Verify the internal consistency of recorded `ROI` values against raw financial metrics (`Revenue` and `Spend`) to identify miscalculated or erroneous reporting records.

**Mathematical Formula:**  
$$\text{Calculated ROI} = \frac{\text{Revenue} - \text{Spend}}{\text{Spend}}$$

**Methodology Applied:**  
1. **Independent ROI Computation:** Calculated baseline ROI from `Revenue` and `Spend` columns, rounding results to two decimal places.
2. **Tolerance-Based Mismatch Mask:** Flagged records where both `Revenue` and `Spend` are non-null and the absolute difference between recorded `ROI` and calculated ROI exceeds a tolerance threshold ($|\text{ROI} - \text{ROI}_{\text{calc}}| > 0.02$) to account for minor rounding variance.
3. **Record Inspection:** Isolated discrepant campaign records alongside their identifiers and notes for auditing.

In [28]:
roi_calc = round((df["Revenue"] - df["Spend"])/ df["Spend"], 2)
roi_mismatch = df["Revenue"].notna() & df["Spend"].notna() & ((df["ROI"] - roi_calc).abs() > 0.02)
df.loc[roi_mismatch, ["CampaignID", "Revenue", "Spend", "ROI", "Notes"]]

,CampaignID,Revenue,Spend,ROI,Notes
8,CMP3009,2169.15,987.96,1.7,ROI looks off?


## Reconciling & Overwriting Inconsistent ROI Values

**Objective:**  
Correct discrepancy errors and impute missing Return on Investment figures using direct calculation from verified `Revenue` and `Spend` columns.

**Correction & Verification Applied:**  
1. **Accurate Computation:** Computed true ROI values using $\frac{\text{Revenue} - \text{Spend}}{\text{Spend}}$, rounded to two decimal places, and staged them into `ROI_Fixed`.
2. **Column Reconciliation:** Updated the primary `ROI` column with the computed values, falling back to original records via `.fillna()` for rows lacking sufficient spend/revenue data.
3. **Post-Correction Audit:** Re-evaluated the tolerance mismatch filter ($|\text{ROI} - \text{ROI}_{\text{calc}}| > 0.02$) to verify that all arithmetic discrepancies across complete records are resolved.

In [29]:
df["ROI_Fixed"] = round((df["Revenue"] - df["Spend"])/ df["Spend"], 2)

df["ROI"] = df["ROI_Fixed"].fillna(df["ROI"])

In [30]:
roi_mismatch = df["Revenue"].notna() & df["Spend"].notna() & ((df["ROI"] - roi_calc).abs() > 0.02)
df.loc[roi_mismatch, ["CampaignID", "Revenue", "Spend", "ROI", "Notes"]]

,CampaignID,Revenue,Spend,ROI,Notes


In [31]:
df.loc[df["CampaignID"] == "CMP3009", "Notes"] = np.nan

## CTR Validation, Recalculation & Note Clearance

**Objective:**  
Verify reported Click-Through Rate (`CTR`) figures against raw traffic engagement metrics (`Clicks` and `Impressions`), correct computational errors, and clear out stale error notes once resolved.

**Mathematical Formula:**  
$$\text{Calculated CTR (\%)} = \left(\frac{\text{Clicks}}{\text{Impressions}}\right) \times 100$$

**Methodology & Fix Applied:**  
1. **Safe CTR Computation:** Calculated expected CTR percentages while replacing zero-impression denominators with `np.nan` to prevent division-by-zero errors, rounding to two decimal places.
2. **Tolerance-Based Discrepancy Detection:** Filtered for valid non-null rows where the recorded and calculated CTR deviated by more than a tolerance threshold ($|\text{CTR} - \text{CTR}_{\text{calc}}| > 0.05$).
3. **Metric Reconciliation:** Updated the primary `CTR` column using calculated values (`CTR_Calc`), falling back to existing entries via `.fillna()` where impression data was unavailable.
4. **Audit Trail Reset:** Reset the `Notes` field to `np.nan` for reconciled mismatch rows to clear resolved flag comments.

In [32]:
ctr_calc = round(df["Clicks"] / df["Impressions"].replace(0, np.nan) * 100, 2)
ctr_mismatch = df["Clicks"].notna() & df["Impressions"].notna() & ((df["CTR"] - ctr_calc).abs() > 0.05)
df.loc[ctr_mismatch, ["CampaignID", "Clicks", "Impressions", "CTR", "Notes"]]

df["CTR_Calc"] = round(df["Clicks"] / df["Impressions"].replace(0, np.nan) * 100, 2)
df["CTR"] = df["CTR_Calc"].fillna(df["CTR"])
df.loc[ctr_mismatch, "Notes"] = np.nan

## Landing Page URL Verification

**Objective:**  
Identify broken, incomplete, or malformed destination URLs in the `LandingPageURL` column using regular expression matching.

**Validation Pattern:**  
`^https://[a-zA-Z0-9\-\.]+\.[a-zA-Z]{2,}(/[^\s]*)?$`
* **Protocol:** Strictly enforces secure HTTPS protocol (`https://`).
* **Domain & Subdomains:** Matches standard alphanumeric domain names, hyphens, subdomains, and top-level domain extensions of at least 2 characters (`.com`, `.org`, etc.).
* **Path & Query Strings:** Validates optional non-whitespace resource paths, query parameters, or route endpoints.

**Methodology Applied:**  
1. **Regex String Matching:** Applied `.str.match()` on string-cast URL entries to verify adherence to standard HTTPS URL structures.
2. **Invalid URL Isolation:** Used the inverse boolean mask (`~valid_url`) to isolate and inspect flagged non-compliant URLs alongside their `CampaignID` for review.

In [33]:
url_pattern = r"^https://[a-zA-Z0-9\-\.]+\.[a-zA-Z]{2,}(/[^\s]*)?$"
valid_url = df["LandingPageURL"].astype(str).str.match(url_pattern)
df.loc[~valid_url, ["CampaignID", "LandingPageURL"]]

,CampaignID,LandingPageURL
2,CMP3003,htp://example.com/typo
10,CMP3011,www.example.com/no-protocol
19,CMP3020,https://example .com/space
25,CMP3026,ftp://example.com/wrong-scheme


## `Channel`, `Device`, `Status`

In [34]:
chan_map = {"paid search": "Paid Search", "social": "Social", "display": "Display", "email": "Email"}
df["Channel"] = df["Channel"].str.strip().str.lower().map(chan_map)
df["Device"] = df["Device"].str.strip().str.title()
df["Status"] = df["Status"].str.strip().str.title()
df[["Channel", "Device", "Status"]].drop_duplicates().head()

,Channel,Device,Status
0,Email,Mobile,Paused
1,Social,Desktop,Paused
2,Email,Tablet,Active
3,Email,Desktop,Paused
4,Social,Tablet,Completed


In [35]:
df["Notes"] = df["Notes"].astype(str).str.strip()
df["Notes"] = df["Notes"].replace(["", "nan", "None"], np.nan)

In [36]:
dup_cols = ["CampaignName", "Country", "Impressions", "Clicks", "Channel"]
df[df.duplicated(subset=dup_cols, keep=False)][["CampaignID","CampaignName","Channel","Notes"]]

,CampaignID,CampaignName,Channel,Notes
7,CMP3008,Product Launch,Paid Search,NaN
12,CMP3013,Cart Abandon Recovery,Paid Search,NaN
40,CMP3041,Product Launch,Paid Search,NaN
44,CMP3045,Cart Abandon Recovery,Paid Search,Possible duplicate


In [37]:
df = df.drop_duplicates(subset=dup_cols, keep="first").reset_index(drop=True)
df.shape

(46, 19)

In [38]:
df.dtypes

CampaignID                        str
CampaignName                      str
Channel                           str
Country                           str
StartDate         datetime64[us, UTC]
EndDate           datetime64[us, UTC]
Impressions                     int64
Clicks                          int64
CTR                           float64
Spend                         float64
Revenue                       float64
ROI                           float64
LandingPageURL                    str
Device                            str
Status                            str
Notes                             str
NeedsReview                    object
ROI_Fixed                     float64
CTR_Calc                      float64
dtype: object

In [39]:
df.to_csv("Cleaned_campaigns_data.csv", index = False)